# Create 100 m template and dataset
Full pipeline: build `template_100m.pkl` with finest scale at index 0 → save train/test pkl files.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

## Config

In [ ]:
# Template output path
TEMPLATE_PKL = 'database/datasets/train/template_100m.pkl'

# SFINCS 100 m map used for the finest mesh level of the template
SFINCS_MAP_TEMPLATE = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart/sfincs_map.nc'
)

# Non-warmstart sfincs_map.nc — same grid, stores x/y as 2D arrays (used for source point coordinates)
SFINCS_MAP_GRID = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon/sfincs_map.nc'
)

# GIS files (boundary polygon + DEM) — from non-warmstart run, identical domain geometry
SHAPEFILE = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon/gis/region.geojson'
)
DEM_TIF = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon/gis/dep.tif'
)

# Coarse gmesh resolutions [m] — 3 levels, coarsest to finest
MESH_RESOLUTIONS = [2000, 1000, 500]

# Simulations to convert — each tuple: (sfincs_dir, dataset_name, out_split)
SFINCS_DIR = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart'
)

simulations = [
    (SFINCS_DIR, 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart', 'train'),
    (SFINCS_DIR, 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart', 'test'),
]

OUT_ROOT = 'database/datasets'

# SFINCS variable names
WATER_LEVEL_VAR = 'zs'
BED_LEVEL_VAR   = 'zb'
VX_VAR = 'u'   # set to None if not present
VY_VAR = 'v'   # set to None if not present

# Force flags
FORCE_REBUILD_TEMPLATE = True   # set True to recreate template from scratch
FORCE_REBUILD_PKL      = True   # set True to recreate dataset pkls

## Step 1 — Create template

In [7]:
from database.create_mesh_template_marg import create_mesh_template_pkl

if FORCE_REBUILD_TEMPLATE and os.path.exists(TEMPLATE_PKL):
    os.remove(TEMPLATE_PKL)
    print('Deleted stale template:', TEMPLATE_PKL)

if os.path.exists(TEMPLATE_PKL):
    print('Template already exists:', TEMPLATE_PKL)
    print('Set FORCE_REBUILD_TEMPLATE = True to recreate.')
else:
    create_mesh_template_pkl(
        shapefile_path        = SHAPEFILE,
        dem_tif_path          = DEM_TIF,
        output_pkl_path       = TEMPLATE_PKL,
        with_multiscale       = True,
        number_of_multiscales = 4,
        mesh_resolutions      = MESH_RESOLUTIONS,
        sfincs_map_nc         = SFINCS_MAP_TEMPLATE,
    )
    print('Template saved:', TEMPLATE_PKL)


=== Creating Mesh Template ===
Shapefile: database/raw_datasets_ahr/Simulations/ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart/gis/region.geojson
DEM: database/raw_datasets_ahr/Simulations/ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart/gis/dep.tif
Output: database/datasets/train/template_100m.pkl
Multiscale: True

1. Converting shapefile to polygon...


FileNotFoundError: Shapefile not found: database/raw_datasets_ahr/Simulations/ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart/gis/region.geojson

## Step 3 — Create dataset pkl files

In [ ]:
import pickle
import numpy as np
import torch
import xarray as xr
import copy

from database.convert_sfincs_to_pkl_marg import (
    load_single_data_object,
    get_target_points,
    get_source_points,
    interpolate_time_series,
    parse_src_file,
    parse_dis_file,
    build_output_data,
)

print('Loading template...')
template_data = load_single_data_object(TEMPLATE_PKL)
target_points = get_target_points(template_data)
print('  Template mesh faces:', target_points.shape[0])

# Get grid coordinates from non-warmstart file (2D x/y, same grid as warmstart)
ds_grid = xr.open_dataset(SFINCS_MAP_GRID, decode_times=False)
source_points = get_source_points(ds_grid)
ds_grid.close()
print('  Source points:', source_points.shape[0])

for sfincs_dir, dataset_name, out_split in simulations:
    out_path = os.path.join(OUT_ROOT, out_split, dataset_name + '.pkl')
    if not FORCE_REBUILD_PKL and os.path.exists(out_path):
        print('Skipping (already exists):', out_path)
        continue

    sfincs_map = os.path.join(sfincs_dir, 'sfincs_map.nc')
    src_file   = os.path.join(sfincs_dir, 'sfincs.src')
    dis_file   = os.path.join(sfincs_dir, 'sfincs.dis')

    print()
    print('Processing:', dataset_name, '->', out_split)
    ds = xr.open_dataset(sfincs_map, decode_times=False)

    zs = ds[WATER_LEVEL_VAR].values
    zb = ds[BED_LEVEL_VAR].values
    WD_grid = np.maximum(zs - zb[None, :, :], 0.0).astype(np.float32)
    print('  Interpolating WD...')
    WD = interpolate_time_series(source_points, WD_grid, target_points, 'WD')

    ds_raw = xr.open_dataset(sfincs_map, decode_times=False, mask_and_scale=False)
    if VX_VAR and VX_VAR in ds.data_vars:
        VX_raw = ds_raw[VX_VAR].values.astype(np.float32)
        fv = ds_raw[VX_VAR].attrs.get('_FillValue', None)
        if fv is not None:
            VX_raw[VX_raw == fv] = np.nan
        print('  Interpolating VX...')
        VX = interpolate_time_series(source_points, VX_raw, target_points, 'VX')
    else:
        VX = np.zeros_like(WD)
    if VY_VAR and VY_VAR in ds.data_vars:
        VY_raw = ds_raw[VY_VAR].values.astype(np.float32)
        fv = ds_raw[VY_VAR].attrs.get('_FillValue', None)
        if fv is not None:
            VY_raw[VY_raw == fv] = np.nan
        print('  Interpolating VY...')
        VY = interpolate_time_series(source_points, VY_raw, target_points, 'VY')
    else:
        VY = np.zeros_like(WD)
    ds_raw.close()

    time_var = ds.coords.get('time', ds.coords.get('t', None))
    map_times_s = (time_var.values.astype(np.float64) if time_var is not None
                   else np.arange(zs.shape[0]) * 3600.0)
    ds.close()

    print('  Reading src/dis files...')
    src_xy = parse_src_file(src_file)
    dis_times_s, discharge = parse_dis_file(dis_file)
    print(' ', len(src_xy), 'source points, discharge shape:', discharge.shape)

    data_out = build_output_data(
        template_data, WD=WD, VX=VX, VY=VY,
        map_times_s=map_times_s, src_xy=src_xy,
        dis_times_s=dis_times_s, discharge=discharge,
    )

    os.makedirs(os.path.join(OUT_ROOT, out_split), exist_ok=True)
    with open(out_path, 'wb') as f:
        pickle.dump([data_out], f)
    print('  Saved:', out_path)
    print('  WD=', tuple(data_out.WD.shape),
          '| node_BC=', data_out.node_BC.tolist(),
          '| BC=', tuple(data_out.BC.shape))